# 1D RWKV ground-state search

Small transverse-field Ising example using the imported autoregressive RWKV ansatz.

In [ ]:
import jax
import jax.numpy as jnp

import jVMC_exp
import jVMC_exp.nets as nets
import jVMC_exp.operator.discrete as op
import jVMC_exp.sampler as sampler
from jVMC_exp.vqs import NQS

jax.config.update("jax_enable_x64", True)

In [ ]:
L = 4
g = 1.0

hamiltonian = 0
for i in range(L):
    hamiltonian += -op.SigmaZ(i) * op.SigmaZ((i + 1) % L)
    hamiltonian += -g * op.SigmaX(i)

In [ ]:
net = nets.RWKVCPM(
    L=L,
    LHilDim=2,
    patch_size=1,
    hidden_size=8,
    num_heads=1,
    num_layers=1,
    embedding_size=8,
    param_dtype=jnp.float32,
    compute_dtype=jnp.float32,
)
psi = NQS(net, L, batchSize=2**L, seed=1234)
exact_sampler = sampler.ExactSampler(psi)
print("Number of parameters:", psi.numParameters)

In [ ]:
loss_function = jVMC_exp.objective_function.Observable(hamiltonian)
stepper = jVMC_exp.stepper.Euler(2e-2)
opt = jVMC_exp.optimizer.MinSR(exact_sampler, psi, diagonalShift=1e-3, pinv_tol=1e-8, full_batched=True)

out = opt.ground_state_search(20, loss_function, stepper)
energy = exact_sampler(hamiltonian).mean
print("Final energy:", energy)